In [1]:
import math                                                         # pour les calculs d'entropie (log2)
import json                                                         # pour afficher l'arbre proprement

data = [
    ("Sunny",    "Hot",  "High",   "Weak",   "No"),               # jour 1
    ("Sunny",    "Hot",  "High",   "Strong", "No"),               # jour 2
    ("Overcast", "Hot",  "High",   "Weak",   "Yes"),              # jour 3
    ("Rain",     "Mild", "High",   "Weak",   "Yes"),              # jour 4
    ("Rain",     "Cool", "Normal", "Weak",   "Yes"),              # jour 5
    ("Rain",     "Cool", "Normal", "Strong", "No"),               # jour 6
    ("Overcast", "Cool", "Normal", "Strong", "Yes"),              # jour 7
    ("Sunny",    "Mild", "High",   "Weak",   "No"),               # jour 8
    ("Sunny",    "Cool", "Normal", "Weak",   "Yes"),              # jour 9
    ("Rain",     "Mild", "Normal", "Weak",   "Yes"),              # jour 10
    ("Sunny",    "Mild", "Normal", "Strong", "Yes"),              # jour 11
    ("Overcast", "Mild", "High",   "Strong", "Yes"),              # jour 12
    ("Overcast", "Hot",  "Normal", "Weak",   "Yes"),              # jour 13
    ("Rain",     "Mild", "High",   "Strong", "No"),               # jour 14
]

# ──────────────────────────────────────────────
# ÉTAPE 1 : Calculer l'entropie d'un groupe
# Entropie mesure le désordre : 0 = pur (tout Yes ou tout No), 1 = maximum de désordre
# Formule : H = -(p_yes * log2(p_yes)) - (p_no * log2(p_no))
# ──────────────────────────────────────────────
def entropie(dataset):
    h, nbr_yes, nbr_no = 0, 0, 0                                  # initialiser entropie et compteurs à 0
    for i in dataset:                                              # parcourir chaque ligne du dataset
        if i[4] == 'Yes':                                         # si la classe est Yes
            nbr_yes = nbr_yes + 1                                 # incrémenter le compteur Yes
        else:                                                      # sinon
            nbr_no = nbr_no + 1                                   # incrémenter le compteur No
    if nbr_yes == 0 or nbr_no == 0:                               # si un seul groupe existe (noeud pur)
        return 0                                                   # entropie = 0 (pas de désordre)
    p_yes = nbr_yes / len(dataset)                                # probabilité d'avoir Yes
    p_no  = nbr_no  / len(dataset)                                # probabilité d'avoir No
    h = -(p_yes * math.log2(p_yes)) - (p_no * math.log2(p_no))   # formule de l'entropie de Shannon
    return h                                                       # retourner l'entropie calculée

# ──────────────────────────────────────────────
# ÉTAPE 2 : Calculer l'entropie conditionnelle d'un attribut
# On divise le dataset selon chaque valeur de l'attribut et on fait la moyenne pondérée
# ──────────────────────────────────────────────
def entropie_attribut(dataset,colon):
    totale=len(dataset)                                            # nombre total de lignes
    h_attribut = 0                                                 # entropie conditionnelle initialisée à 0
    valeurs=set(row[colon] for row in dataset)                     # Étape 1 : quelles valeurs existe dans cette colonne ?
    for i in valeurs:                                              # pour chaque valeur unique de l'attribut
        subset=[row for row in dataset if row[colon]==i]           # Étape 2 : garder seulement les lignes où col == i
        poids=len(subset)/totale                                   # Étape 3 : calculer le poids de ce groupe
        h_attribut += poids * entropie(subset)                    # Étape 4 : ajouter la contribution de ce groupe
    return h_attribut                                             # retourner l'entropie conditionnelle totale

# ──────────────────────────────────────────────
# ÉTAPE 3 : Calculer le gain d'information d'un attribut
# Gain = entropie du dataset entier - entropie conditionnelle de l'attribut
# ──────────────────────────────────────────────
def gain(dataset, col):
    return entropie(dataset) - entropie_attribut(dataset, col)    # gain = H(dataset) - H(dataset | attribut)

# ──────────────────────────────────────────────
# ÉTAPE 4 : Trouver le meilleur attribut
# On teste TOUS les attributs disponibles et on garde celui avec le gain le plus élevé
# ──────────────────────────────────────────────
def meilleur_attribut(dataset, attributs):
    meilleur_gain=-1                                               # on commence avec un gain négatif
    meilleur_attr=None                                             # attribut du meilleur gain
    for col in attributs:                                          # pour chaque attribut disponible
        g = gain(dataset, col)                                     # calculer le gain de cet attribut
        print(f"  Gain(col={col}) = {g:.4f}")                     # afficher pour visualiser
        if g > meilleur_gain:                                      # si c'est le meilleur jusqu'ici
            meilleur_gain = g                                      # sauvegarder le gain
            meilleur_attr = col                                    # sauvegarder l'attribut
    return meilleur_attr                                           # retourner le meilleur attribut trouvé

# ──────────────────────────────────────────────
# ÉTAPE 5 : Algorithme ID3
# Construit un arbre multi-branches récursivement en choisissant le meilleur attribut à chaque noeud
# ──────────────────────────────────────────────
def id3(dataset,attributs):
   # ── Cas 1 : tout Yes ?
    if all(row[4] == "Yes" for row in dataset):                    # si tous les exemples sont Yes
        return "Yes"                                               # feuille = Yes
   # ── Cas 1 : tout Non ?
    if all(row[4] == "Non" for row in dataset):                    # si tous les exemples sont Non
        return "Non"                                               # feuille = Non
     # ── Cas 3 : plus d'attributs → classe majoritaire
    if len(attributs) == 0:                                        # si plus aucun attribut disponible
        nbr_yes = sum(1 for row in dataset if row[4] == "Yes")     # compter les Yes
        nbr_no  = len(dataset) - nbr_yes                          # compter les No
        return "Yes" if nbr_yes >= nbr_no else "No"               # retourner la classe majoritaire
   # ── Cas 4 : choisir le meilleur attribut
    best_col = meilleur_attribut(dataset, attributs)               # trouver l'attribut avec le meilleur gain
    arbre = {best_col: {}}                                         # créer le noeud de l'arbre avec cet attribut
    attributs_restants = [a for a in attributs if a != best_col]   # retirer best_col des attributs restants
    valeurs = set(row[best_col] for row in dataset)                # toutes les valeurs possibles de best_col
    for v in valeurs:                                              # pour chaque valeur possible
        subset = [row for row in dataset if row[best_col] == v]    # garder seulement les lignes où best_col == v
        arbre[best_col][v] = id3(subset, attributs_restants)       # appel récursif sur ce sous-groupe
    return arbre                                                   # retourner le noeud complet

# ──────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────
print(entropie(data))                                              # afficher l'entropie initiale du dataset
print(entropie_attribut(data, 1))                                  # afficher l'entropie conditionnelle de la colonne 1 (Temperature)
attributs = [0, 1, 2, 3]                                          # Outlook, Temperature, Humidity, Wind
best = meilleur_attribut(data, attributs)                          # trouver le meilleur attribut à la racine
print(f"\nMeilleur attribut : colonne {best}")                     # afficher le meilleur attribut trouvé
import json                                                        # déjà importé mais replacé ici dans le code original
attributs = [0, 1, 2, 3]                                          # réinitialiser la liste des attributs
arbre = id3(data, attributs)                                       # construire l'arbre ID3 complet
print(json.dumps(arbre, indent=4))                                 # afficher l'arbre final

0.9402859586706311
0.9110633930116763
  Gain(col=0) = 0.2467
  Gain(col=1) = 0.0292
  Gain(col=2) = 0.1518
  Gain(col=3) = 0.0481

Meilleur attribut : colonne 0
  Gain(col=0) = 0.2467
  Gain(col=1) = 0.0292
  Gain(col=2) = 0.1518
  Gain(col=3) = 0.0481
  Gain(col=1) = 0.5710
  Gain(col=2) = 0.9710
  Gain(col=3) = 0.0200
  Gain(col=1) = 0.0000
  Gain(col=3) = 0.0000
  Gain(col=3) = 0.0000
  Gain(col=3) = 0.0000
  Gain(col=1) = 0.0200
  Gain(col=2) = 0.0200
  Gain(col=3) = 0.9710
  Gain(col=1) = 0.0000
  Gain(col=2) = 0.0000
  Gain(col=2) = 0.0000
  Gain(col=2) = 0.0000
{
    "0": {
        "Overcast": "Yes",
        "Sunny": {
            "2": {
                "Normal": "Yes",
                "High": {
                    "1": {
                        "Mild": {
                            "3": {
                                "Weak": "No"
                            }
                        },
                        "Hot": {
                            "3": {
                      